In [ ]:
# grid_hist_all_metrics_uniform.py
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import LinearLocator
from matplotlib.ticker import LinearLocator, FuncFormatter


In [ ]:
SIGFIGS_Y = 3

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman No9 L", "DejaVu Serif"],  # fallbacks
    "mathtext.fontset": "dejavuserif",   # math text harmonizes with serif
    "axes.unicode_minus": False,         # proper minus sign with serif fonts
    "pdf.fonttype": 42,                  # embed TrueType in vector outputs
    "ps.fonttype": 42,
})

# ---------- Paths ----------
IN_DIR  = "final_result"                               # where Aggregated_Summary_Core{i}.csv live
OUT_DIR = os.path.join(IN_DIR, "heldout_lines_optionB")# keep outputs near your other figures
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Cores & groups ----------
CORES = list(range(1, 9))                              # Core1..Core8
GROUP_ORDER = [f"{k} core" if k == 1 else f"{k} cores" for k in range(1, 8)]
CORE_LABELS = [f"Core {c}" for c in CORES]

# ---------- Which metrics go in which figure (Display name, key in CSV) ----------
FIG1_METRICS = [
    ("MAE",   "mae"),
    ("GMSD",  "gmsd"),
    ("LPIPS", "lpips"),
    ("DISTS", "dists"),
]
FIG2_METRICS = [
    ("R2",         "r2"),
    ("Soft Dice",  "soft_dice"),
    ("PSNR",       "psnr"),
    ("VIF",        "vif"),
    ("SSIM",       "ssim"),
    ("MS-SSIM",    "ms_ssim"),
    ("FSIM",       "fsim"),
]

def make_sigfig_formatter(sig=3):
    """Plain-decimal tick labels with `sig` significant digits, no sci-notation."""
    def _fmt(y, pos=None):
        if not np.isfinite(y) or y == 0:
            return "0"
        sgn = "-" if y < 0 else ""
        y = abs(y)
        dec = sig - int(np.floor(np.log10(y))) - 1
        y_rounded = round(y, dec)
        dec_out = max(0, dec)
        s = f"{y_rounded:.{dec_out}f}"
        if "." in s:
            s = s.rstrip("0").rstrip(".")
        return sgn + s
    return FuncFormatter(_fmt)
    
# ---------- Load aggregated summaries ----------
def load_all_aggregated():
    dfs = []
    for i in CORES:
        p = os.path.join(IN_DIR, f"Aggregated_Summary_Core{i}.csv")
        if not os.path.exists(p):
            print(f"[WARN] Missing: {p} (skip Core {i})")
            continue
        df = pd.read_csv(p)
        df["core"] = i
        dfs.append(df)
    if not dfs:
        raise FileNotFoundError("No Aggregated_Summary_Core{i}.csv files found in final_result/")
    all_df = pd.concat(dfs, ignore_index=True)
    # normalize fields
    all_df["metric"] = all_df["metric"].str.lower().str.strip()
    all_df["group"]  = all_df["group"].str.lower().str.strip()
    # keep only expected groups
    all_df = all_df[all_df["group"].isin(GROUP_ORDER)]
    return all_df

def matrix_group_x_core(sub_df, value_col):
    """
    Returns a DataFrame with rows=GROUP_ORDER and cols='Core 1'..'Core 8'
    Each cell is sub_df[value_col] for that (group, core).
    """
    mat = pd.DataFrame(index=GROUP_ORDER, columns=[f"Core {c}" for c in CORES], dtype=float)
    for c in CORES:
        tmp = sub_df[sub_df["core"] == c][["group", value_col]].dropna()
        if not tmp.empty:
            s = tmp.set_index("group")[value_col]
            mat[f"Core {c}"] = s.reindex(GROUP_ORDER)
    return mat

# ---------- Shared plotting routine ----------
def draw_metric_grid(df, metric_list, out_png, dpi=220):
    """
    metric_list: list of (display_name, key)
    Produces a grid with rows=len(metric_list), cols=2 (mean & std across models).
    """
    n_rows = len(metric_list)
    n_cols = 2

    # choose colors (consistent across all subplots)
    color_cycle = list(plt.get_cmap("tab10").colors)
    colors = (color_cycle * ((len(CORES) // len(color_cycle)) + 1))[:len(CORES)]

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3*n_rows), dpi=dpi)
    if n_rows == 1:
        axes = np.array([axes])  # ensure 2D array

    # column headers
    #axes[0, 0].set_title("Mean across CT images", fontsize=15)
    #axes[0, 1].set_title("Std across models", fontsize=15)

    # For a single shared legend (one per figure)
    sample_handles = None

    for r_idx, (disp_name, key) in enumerate(metric_list):
        sub = df[df["metric"] == key].copy()

        # ---- Left column: mean_avg ----
        axL = axes[r_idx, 0]
        if "mean_avg" in sub.columns and not sub.empty:
            mat_mean = matrix_group_x_core(sub, "mean_avg")
            x = np.arange(len(GROUP_ORDER))
            handles = []
            for ci, col in enumerate(mat_mean.columns):
                y = mat_mean[col].values.astype(float)
                h, = axL.plot(x, y, marker='o', linewidth=1.6, label=col, color=colors[ci])
                handles.append(h)
            axL.set_xticks(x)
            axL.set_xticklabels(GROUP_ORDER, rotation=0, fontsize=14)
            axL.set_ylabel(disp_name, fontsize=14)
            if r_idx < n_rows - 1:
                axL.set_xticklabels([])  # hide interior x labels to reduce clutter
            if sample_handles is None:  # capture lines for legend once
                sample_handles = handles
        else:
            axL.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=14)
            axL.set_ylabel(disp_name, fontsize=14)
        axL.yaxis.set_major_locator(LinearLocator(4))
        axL.yaxis.set_major_formatter(make_sigfig_formatter(SIGFIGS_Y))
        
        # ---- Right column: mean_std_across_models ----
        axR = axes[r_idx, 1]
        if "mean_std_across_models" in sub.columns and not sub.empty:
            mat_std = matrix_group_x_core(sub, "mean_std_across_models")
            x = np.arange(len(GROUP_ORDER))
            for ci, col in enumerate(mat_std.columns):
                y = mat_std[col].values.astype(float)
                axR.plot(x, y, marker='o', linewidth=1.6, label=col, color=colors[ci])
            axR.set_xticks(x)
            axR.set_xticklabels(GROUP_ORDER, rotation=0, fontsize=14)
            if r_idx < n_rows - 1:
                axR.set_xticklabels([])
        else:
            axR.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=14)
        
        ymin, ymax = axR.get_ylim()
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = 1.0
        axR.set_ylim(0, ymax)

        axR.yaxis.set_major_locator(LinearLocator(4))
        axR.yaxis.set_major_formatter(make_sigfig_formatter(SIGFIGS_Y))
    # Put a single legend at the bottom (Core 1..8)
    if sample_handles:
        fig.legend(sample_handles, CORE_LABELS, ncol=8, loc="upper center", fontsize=14, frameon=False)

    fig.tight_layout(rect=[0.02, 0.06, 1, 0.96])
    fig.savefig(out_png)
    plt.close(fig)
    print(f"[SAVE] {out_png}")

# ---------- Main ----------
def main():
    df = load_all_aggregated()

    # Figure A: 4×2 (MAE, GMSD, LPIPS, DISTS)
    out_png_a = os.path.join(OUT_DIR, "LINES_4x2_MAE_GMSD_LPIPS_DISTS.png")
    draw_metric_grid(df, FIG1_METRICS, out_png_a, dpi=220)

    # Figure B: 7×2 (R2, Soft Dice, PSNR, VIF, SSIM, MS-SSIM, FSIM)
    out_png_b = os.path.join(OUT_DIR, "LINES_7x2_R2_SoftDice_PSNR_VIF_SSIM_MSSSIM_FSIM.png")
    draw_metric_grid(df, FIG2_METRICS, out_png_b, dpi=220)



In [ ]:
main()